# Conditional Edges in LangGraph: Practice Exercise

Build a support ticket routing system that uses conditional edges to dynamically route tickets to different handlers based on their priority level.

**What you'll implement:**
1. An analyzer node that uses an LLM with structured output to classify ticket priority
2. A routing function that determines the next node based on the classified priority
3. The conditional edge that connects the analyzer to the appropriate handler

**Skills practiced:**
- Using structured output with Pydantic models (from Level 2)
- Creating routing functions for conditional edges
- Adding conditional edges to LangGraph workflows

In [1]:
!pip install langchain-core langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 13.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19


In [2]:
import os

def load_api_key():
  IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ


  if not IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv()
    # Verify OpenAI API key is set
    if not os.getenv("OPENAI_API_KEY"):
      ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    from google.colab import userdata
    OPENAI_API_KEY=userdata.get('OPEN_AI_API_KEY')
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [3]:
load_api_key()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

## Setup

Run this cell to import all required libraries and configure the environment.

In [4]:
# Setup - run this cell first

from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END


print("Setup complete!")

Setup complete!


## Context

You are building a support ticket routing system for a software company. When a ticket comes in, it gets analyzed for priority using an LLM, then routed to the appropriate handler:

- **Critical** tickets (system down, security issues) go to the emergency response team
- **High** priority tickets (bugs affecting users) go to senior engineers  
- **Normal** priority tickets (feature requests, general questions) go to the standard support queue

**Input:** A ticket with description text and empty priority/response fields

**Output:** The ticket with LLM-classified priority and appropriate response message

**Workflow structure:**
```
START -> analyzer (LLM classifies priority) -> [conditional routing] -> handler -> END
```

## Provided Code: State, Schema, and Handler Nodes

The state structure, structured output schema, and handler node functions are provided for you. Review them to understand the workflow.

In [5]:
# State definition - provided for you

class TicketState(TypedDict):
    """State structure for the ticket routing workflow."""
    ticket_description: str   # The original ticket text
    priority: str             # Classified priority: 'critical', 'high', or 'normal'
    response: str             # Handler's response message

print("State defined!")

State defined!


In [6]:
# Structured output schema - provided for you

class TicketClassification(BaseModel):
    """Structured output schema for ticket priority classification."""

    priority: Literal["critical", "high", "normal"] = Field(
        description="The priority level of the ticket. "
        "Use 'critical' for system outages, security breaches, or service down. "
        "Use 'high' for bugs, errors, or broken functionality. "
        "Use 'normal' for feature requests, questions, or general inquiries."
    )

print("TicketClassification schema defined!")

TicketClassification schema defined!


In [7]:
# Handler node functions - provided for you

def emergency_handler_node(state: TicketState) -> TicketState:
    """
    Handles critical priority tickets.
    """
    print("Emergency Handler: Processing critical ticket...")
    response = f"CRITICAL ALERT: Emergency team engaged for: '{state['ticket_description']}'. Incident response initiated."

    return {
        "ticket_description": state["ticket_description"],
        "priority": state["priority"],
        "response": response
    }


def senior_handler_node(state: TicketState) -> TicketState:
    """
    Handles high priority tickets.
    """
    print("Senior Handler: Processing high priority ticket...")
    response = f"HIGH PRIORITY: Senior engineer assigned to: '{state['ticket_description']}'. Expected resolution within 4 hours."

    return {
        "ticket_description": state["ticket_description"],
        "priority": state["priority"],
        "response": response
    }


def standard_handler_node(state: TicketState) -> TicketState:
    """
    Handles normal priority tickets.
    """
    print("Standard Handler: Processing normal ticket...")
    response = f"NORMAL: Ticket queued for: '{state['ticket_description']}'. Expected response within 24 hours."

    return {
        "ticket_description": state["ticket_description"],
        "priority": state["priority"],
        "response": response
    }

print("Handler node functions defined!")

Handler node functions defined!


---

## Part 1: Implement the Analyzer Node

Create the analyzer node that uses an LLM with structured output to classify the ticket priority.

**Requirements:**
1. Create a `ChatOpenAI` model instance
2. Use `with_structured_output()` to bind the `TicketClassification` schema
3. Invoke the model with a prompt that includes the ticket description
4. Extract the priority from the structured response and update the state

**Hint:** The structured model returns a `TicketClassification` object. Access the priority with `result.priority`.

In [8]:
def analyzer_node(state: TicketState) -> TicketState:
    """
    Analyzes the ticket using an LLM and classifies its priority level.

    Uses structured output to ensure the LLM returns a valid priority value.

    Args:
        state: The current TicketState containing the ticket_description

    Returns:
        Updated TicketState with the priority field set
    """
    # TODO: Implement the analyzer node

    # Step 1: Create a ChatOpenAI model (use gpt-4o-mini for efficiency)
    model = ChatOpenAI(model="gpt-4o-mini")
    # Step 2: Create a structured output model using with_structured_output()
    #         Pass the TicketClassification schema
    structure_model = model.with_structured_output(TicketClassification)
    # Step 3: Create a prompt that asks the LLM to classify the ticket
    #         Include the ticket description from state["ticket_description"]
    prompt = f"Classify the priority of the support ticket: {state['ticket_description']}"
    # Step 4: Invoke the structured model with the prompt
    result  = structure_model.invoke(prompt)
    # Step 5: Extract the priority from the result
    priority = result.priority
    # Step 6: Print the classification for visibility
    #         print(f"Analyzer: Ticket classified as '{priority}' priority")
    print(f"Analyzer: Ticket classified as '{priority}' priority")
    # Step 7: Return the updated state with the new priority
    return{
        "ticket_description": state["ticket_description"],
        "priority": priority,
        "response": state.get('response',"")
    }

print("Analyzer node defined!")

Analyzer node defined!


---

## Part 2: Implement the Routing Function

Create a routing function that examines the state and returns a string indicating which handler should process the ticket.

**Requirements:**
- Read the `priority` field from the state
- Return `"critical"` if priority is "critical"
- Return `"high"` if priority is "high"
- Return `"normal"` for all other cases

**Note:** The return value will be mapped to node names via the path map in Part 3.

In [9]:
def route_ticket(state: TicketState) -> Literal["critical", "high", "normal"]:
    """
    Routing function that determines which handler processes the ticket.

    Examines the 'priority' field in the state and returns a string that
    will be mapped to the appropriate handler node via the path map.

    Args:
        state: The current TicketState containing the priority field

    Returns:
        One of: "critical", "high", or "normal"
    """
    # TODO: Implement the routing logic
    # 1. Get the priority from state
    # 2. Print the routing decision for visibility:
    #    print(f"Router: Routing to '{priority}' handler")
    # 3. Return the appropriate string based on priority value
    priority = state.get("priority","")
    print(f"Router: Routing to '{priority}' handler")
    if priority == "critical":
        return "critical"
    elif priority == "high":
        return "high"
    else:
        return "normal"

print("Routing function defined!")

Routing function defined!


---

## Part 3: Add the Conditional Edge to the Workflow

The workflow structure is provided below. Your task is to add the **conditional edge** that connects the analyzer to the appropriate handler based on the routing function.

**Your task:** Replace the `# TODO` comment with a call to `add_conditional_edges()`.

**Requirements:**
- Source node: `"analyzer"`
- Routing function: `route_ticket`
- Path map: `{"critical": "emergency_handler", "high": "senior_handler", "normal": "standard_handler"}`

**Syntax reminder:**
```python
workflow.add_conditional_edges(
    "source_node",      # Where the edge comes from
    routing_function,   # Function that returns the routing key
    {                   # Path map: routing key -> target node
        "key1": "node1",
        "key2": "node2"
    }
)
```

In [10]:
def build_ticket_workflow():
    """
    Build and compile the ticket routing workflow.

    Returns:
        The compiled workflow ready to invoke
    """
    # Create the StateGraph
    workflow = StateGraph(TicketState)

    # Add all nodes
    workflow.add_node("analyzer", analyzer_node)
    workflow.add_node("emergency_handler", emergency_handler_node)
    workflow.add_node("senior_handler", senior_handler_node)
    workflow.add_node("standard_handler", standard_handler_node)

    # Add edge from START to analyzer
    workflow.add_edge(START, "analyzer")

    # TODO: Add conditional edges from "analyzer" to the handlers
    # Use the route_ticket function and the path map:
    # {"critical": "emergency_handler", "high": "senior_handler", "normal": "standard_handler"}
    workflow.add_conditional_edges(
        "analyzer",
        route_ticket,
        {
            "critical": "emergency_handler",
            "high": "senior_handler",
            "normal": "standard_handler"
        }

    )

    # Add edges from handlers to END
    workflow.add_edge("emergency_handler", END)
    workflow.add_edge("senior_handler", END)
    workflow.add_edge("standard_handler", END)

    # Compile and return
    return workflow.compile()

print("Workflow builder defined!")

Workflow builder defined!


---

## Run Your Implementation

Test your workflow with different ticket types to verify the conditional routing works correctly.

In [11]:
# Build the workflow
ticket_app = build_ticket_workflow()
print("Workflow built successfully!")

Workflow built successfully!


In [12]:
# Test Case 1: Critical ticket (should route to emergency handler)
print("=" * 60)
print("TEST: Critical Priority Ticket")
print("=" * 60)

critical_ticket = {
    "ticket_description": "Our entire payment system is down and customers cannot complete purchases. This is affecting all users.",
    "priority": "",
    "response": ""
}

result = ticket_app.invoke(critical_ticket)
print(f"\nPriority: {result['priority']}")
print(f"Response: {result['response']}")

TEST: Critical Priority Ticket
Analyzer: Ticket classified as 'critical' priority
Router: Routing to 'critical' handler
Emergency Handler: Processing critical ticket...

Priority: critical
Response: CRITICAL ALERT: Emergency team engaged for: 'Our entire payment system is down and customers cannot complete purchases. This is affecting all users.'. Incident response initiated.


In [13]:
# Test Case 2: High priority ticket (should route to senior handler)
print("=" * 60)
print("TEST: High Priority Ticket")
print("=" * 60)

high_ticket = {
    "ticket_description": "There's a bug in the checkout process. When users apply a discount code, they get an error message and have to restart.",
    "priority": "",
    "response": ""
}

result = ticket_app.invoke(high_ticket)
print(f"\nPriority: {result['priority']}")
print(f"Response: {result['response']}")

TEST: High Priority Ticket
Analyzer: Ticket classified as 'high' priority
Router: Routing to 'high' handler
Senior Handler: Processing high priority ticket...

Priority: high
Response: HIGH PRIORITY: Senior engineer assigned to: 'There's a bug in the checkout process. When users apply a discount code, they get an error message and have to restart.'. Expected resolution within 4 hours.


In [14]:
# Test Case 3: Normal priority ticket (should route to standard handler)
print("=" * 60)
print("TEST: Normal Priority Ticket")
print("=" * 60)

normal_ticket = {
    "ticket_description": "Would it be possible to add dark mode to the settings page? It would be easier on the eyes when working late.",
    "priority": "",
    "response": ""
}

result = ticket_app.invoke(normal_ticket)
print(f"\nPriority: {result['priority']}")
print(f"Response: {result['response']}")

TEST: Normal Priority Ticket
Analyzer: Ticket classified as 'normal' priority
Router: Routing to 'normal' handler
Standard Handler: Processing normal ticket...

Priority: normal
Response: NORMAL: Ticket queued for: 'Would it be possible to add dark mode to the settings page? It would be easier on the eyes when working late.'. Expected response within 24 hours.


---

## Expected Output

If your implementation is correct, you should see:

**Test 1 (Critical):**
- Analyzer classifies as "critical"
- Router routes to "critical" handler
- Emergency Handler processes the ticket

**Test 2 (High):**
- Analyzer classifies as "high"
- Router routes to "high" handler
- Senior Handler processes the ticket

**Test 3 (Normal):**
- Analyzer classifies as "normal"
- Router routes to "normal" handler
- Standard Handler processes the ticket